# Практическое руководство: Перенос данных в SQL и базовые SQL-запросы
## Теория, минимальный набор функций и методика работы

В этом блокноте разобран переход от манипуляций с данными в Pandas к декларативной работе в реляционной СУБД SQLite.

### Содержание:
1. **Зачем нужен SQL аналитику:** концептуальная разница между Pandas и базой данных.
2. **Мостик Pandas ↔ SQLite:** три ключевых метода (`connect`, `to_sql`, `read_sql_query`).
3. **Лекторский минимум операторов фильтрации:** `=`, `!=`, `<`, `>`, `BETWEEN`, `IN`, `LIKE`, `IS NULL`.
4. **Сортировка и пагинация:** `ORDER BY`, `LIMIT`, `OFFSET`.
5. **Работа с датами:** ISO-стандарт `YYYY-MM-DD` и функция `strftime`.
6. **Типичные ошибки студентов:** разбор граблей новичков.


---
## 1. Зачем нужен SQL аналитику данных

| Критерий | Pandas (In-Memory) | База данных / SQL (DBMS) |
|---|---|---|
| **Где живут данные** | В оперативной памяти (RAM) процесса Python | На диске в структурированном хранилище |
| **Ограничение по объему** | Ограничено свободной памятью компьютера | Ограничено только емкостью диска |
| **Парадигма** | Императивная (мы пишем *КАК* пошагово преобразовать) | Декларативная (мы пишем *ЧТО* хотим получить) |
| **Скорость фильтрации** | Приходится загружать весь датасет целиком | СУБД считывает только нужные строки и колонки |

> **Главное правило дата-инженера:** фильтровать, сортировать и агрегировать данные нужно как можно ближе к месту их хранения (в базе данных). В Pandas нужно передавать уже компактный, очищенный результат.


---
## 2. Мостик Python ↔ SQLite

Для локальной работы с реляционными базами данных в стандартную библиотеку Python входит модуль `sqlite3`.
Нам потребуются ровно 3 операции:
1. Подключение: `conn = sqlite3.connect('club_medium.db')`
2. Запись DataFrame в таблицу: `df.to_sql('table_name', conn, if_exists='replace', index=False)`
3. Выполнение запроса и получение DataFrame: `df_res = pd.read_sql_query(sql, conn)`


In [ ]:
import sqlite3
import pandas as pd

# Подключаемся к демонстрационной базе данных компьютерного клуба
conn = sqlite3.connect("club_medium.db")



# Проверим, какие таблицы уже находятся в нашей базе
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Таблицы в базе данных club_medium.db:")
display(tables_df)


Таблицы в базе данных club_medium.db:


,name


---
## 3. Лекторский минимум операторов: SELECT и WHERE

Синтаксический скелет запроса:
```sql
SELECT колонка_1, колонка_2
FROM имя_таблицы
WHERE условие_фильтрации;
```

### 3.1. Операторы сравнения и диапазоны (BETWEEN, IN)
* `=`, `!=` (или `<>`) — проверка на точное равенство/неравенство;
* `<`, `>`, `<=`, `>=` — числовые и строковые сравнения;
* `BETWEEN a AND b` — включительный диапазон `[a; b]`;
* `IN (val1, val2, ...)` — вхождение в перечень допустимых значений.


In [2]:
# Пример 1: Выбор компьютеров зоны Bootcamp с тарифом от 200 до 300 руб.
query_between = """
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE zone_name = 'Bootcamp'
  AND hourly_rate BETWEEN 200 AND 300;
"""

res_between = pd.read_sql_query(query_between, conn)
display(res_between)


DatabaseError: Execution failed on sql '
SELECT pc_id, zone_name, hourly_rate, gpu
FROM hardware_pcs
WHERE zone_name = 'Bootcamp'
  AND hourly_rate BETWEEN 200 AND 300;
': no such table: hardware_pcs

---
### 3.2. Текстовые шаблоны: оператор LIKE

Оператор `LIKE` сопоставляет строки по маске с использованием специальных спецсимволов:
* `%` — любая последовательность символов любой длины (включая 0 символов);
* `_` — ровно один любой символ.

| Шаблон | Что ищет | Пример |
|---|---|---|
| `'RTX 40%'` | Начинается на `RTX 40` | `RTX 4060`, `RTX 4070 Ti`, `RTX 4090` |
| `'%@mail.ru'` | Заканчивается на `@mail.ru` | `gamer123@mail.ru` |
| `'%Strike%'` | Содержит слово `Strike` в любой позиции | `Counter-Strike 2` |
| `'PC-_2'` | Ровно один символ между `PC-` и `2` | `PC-02`, `PC-12` |


In [ ]:
# Пример 2: Клиенты с почтой на домене mail.ru и никнеймом, начинающимся на 'S'
query_like = """
SELECT member_id, nickname, email, loyalty_tier
FROM members
WHERE nickname LIKE 'S%'
  AND email LIKE '%@mail.ru'
LIMIT 5;
"""

res_like = pd.read_sql_query(query_like, conn)
display(res_like)


  member_id      nickname                email loyalty_tier
0    M-1002         Storm        storm@mail.ru         Gold
1    M-1004     Storm_706     storm706@mail.ru       Silver
2    M-1007       Specter      specter@mail.ru         Gold
3    M-1014    Striker_56    striker56@mail.ru       Silver
4    M-1034  Striker_2026  striker2026@mail.ru       Bronze


---
### 3.3. Обработка пропущенных значений: IS NULL / IS NOT NULL

> ⚠️ **Критически важный нюанс SQL:**
> Значение `NULL` обозначает неизвестность. В SQL выражение `email = NULL` **никогда не возвращает True**!
> Для проверки пропусков используются только специальные операторы:
> * `WHERE col IS NULL` — значение отсутствует;
> * `WHERE col IS NOT NULL` — значение заполнено.


In [ ]:
# Пример 3: Поиск клиентов, не указавших контактный email при регистрации
query_null = """
SELECT member_id, nickname, city, registered_at, balance
FROM members
WHERE email IS NULL
LIMIT 5;
"""

res_null = pd.read_sql_query(query_null, conn)
display(res_null)


  member_id      nickname    city registered_at  balance
0    M-1006      Shade_26  Москва    2025-03-27      427
1    M-1009     GhostXx_7  Москва    2025-07-06      431
2    M-1021     BlazeXx_7  Москва    2025-04-17      700
3    M-1026  Specter_2026  Москва    2025-03-10      204
4    M-1029     RoninXx_5  Москва    2025-12-17      244


---
## 4. Сортировка и пагинация: ORDER BY, LIMIT, OFFSET

* `ORDER BY колонка [ASC | DESC]` — сортировка (по умолчанию `ASC` — по возрастанию).
* Можно сортировать сразу по двум столбцам: `ORDER BY zone_name ASC, hourly_rate DESC`.
* `LIMIT n` — взять первые `n` строк.
* `OFFSET m` — предварительно пропустить первые `m` строк.
  * `LIMIT 5 OFFSET 0` — 1-я страница (строки 1..5);
  * `LIMIT 5 OFFSET 5` — 2-я страница (строки 6..10).


In [ ]:
# Пример 4: Топ-3 самых дорогих ПК клуба
query_top = """
SELECT pc_id, zone_name, hourly_rate, gpu, cpu
FROM hardware_pcs
ORDER BY hourly_rate DESC
LIMIT 3;
"""

res_top = pd.read_sql_query(query_top, conn)
display(res_top)


   pc_id zone_name  hourly_rate             gpu              cpu
0  PC-28       VIP          420        RTX 4090  Core i7-13700KF
1  PC-29       VIP          400        RTX 4090  Ryzen 7 7800X3D
2  PC-30       VIP          400  RTX 4080 Super  Core i7-13700KF


In [ ]:
# Пример 5: Пагинация — пропускаем 3 самых дорогих ПК и берем следующие 3
query_page2 = """
SELECT pc_id, zone_name, hourly_rate, gpu, cpu
FROM hardware_pcs
ORDER BY hourly_rate DESC
LIMIT 3 OFFSET 3;
"""

res_page2 = pd.read_sql_query(query_page2, conn)
display(res_page2)


   pc_id zone_name  hourly_rate             gpu              cpu
0  PC-31       VIP          400  RTX 4080 Super  Core i7-13700KF
1  PC-32       VIP          400        RTX 4090  Ryzen 7 7800X3D
2  PC-27       VIP          380  RTX 4080 Super  Core i7-13700KF


---
## 5. Работа с датами: стандарт ISO и функция strftime

В SQLite нет отдельного бинарного типа даты — они хранятся в виде строк стандарта **ISO 8601**:
`YYYY-MM-DD HH:MM:SS` (например, `2026-03-15 18:30:00`).

### Почему именно ISO:
1. Лексикографический порядок строкового сравнения совпадает с хронологическим:
   `'2026-01-15' < '2026-02-01'` — истинно!
2. Для извлечения компонентов даты используется функция:
   `strftime(формат, колонка_даты)`
   * `%Y` — год из 4 цифр (`2026`);
   * `%m` — месяц (`01`..`12`);
   * `%d` — день месяца (`01`..`31`);
   * `%H` — час (`00`..`23`);
   * `%w` — день недели (`0` — воскресенье, `1` — понедельник, ..., `6` — суббота).


In [ ]:
# Пример 6: Фильтрация сессий за март 2026 года с помощью строкового ISO-сравнения
query_dates = """
SELECT session_id, game_title, session_date, hours_played, total_paid
FROM game_sessions
WHERE session_date >= '2026-03-01'
  AND session_date < '2026-04-01'
  AND hours_played >= 4
ORDER BY total_paid DESC
LIMIT 5;
"""

res_dates = pd.read_sql_query(query_dates, conn)
display(res_dates)


   session_id        game_title         session_date  hours_played  total_paid
0        2217            Dota 2  2026-03-11 17:00:00             6        2400
1        2002            Dota 2  2026-03-30 15:45:00             5        2000
2        2021  Counter-Strike 2  2026-03-31 10:45:00             5        2000
3        2484    Cyberpunk 2077  2026-03-18 18:00:00             5        2000
4        2338  Counter-Strike 2  2026-03-24 16:45:00             5        1800


In [ ]:
# Пример 7: Выделение месяца и часа через strftime
query_strftime = """
SELECT 
    session_id, 
    session_date,
    strftime('%Y-%m', session_date) AS session_month,
    strftime('%H', session_date) AS session_hour,
    game_title,
    hours_played
FROM game_sessions
WHERE strftime('%H', session_date) >= '21'
LIMIT 5;
"""

res_strftime = pd.read_sql_query(query_strftime, conn)
display(res_strftime)


   session_id         session_date session_month session_hour        game_title  hours_played
0        2055  2026-03-12 21:15:00       2026-03           21          Valorant             1
1        2064  2026-01-22 21:15:00       2026-01           21             GTA V             2
2        2074  2026-01-20 22:30:00       2026-01           22          EA FC 24             2
3        2079  2026-03-26 23:30:00       2026-03           23            Dota 2             2
4        2084  2026-03-09 21:00:00       2026-03           21  Counter-Strike 2             2


---
## 6. Памятка: типичные ошибки студентов при изучении SQL

| Ошибка студента | В чем причина | Как правильно |
|---|---|---|
| `WHERE email = NULL` | `NULL` обозначает неизвестность, `=` вернет False | `WHERE email IS NULL` |
| `WHERE city = "Москва"` | В стандарте SQL строки берутся в одинарные кавычки | `WHERE city = 'Москва'` |
| `LIMIT 5 OFFSET 1` для 2-й страницы | `OFFSET 1` пропустит всего 1 строку (вместо 5) | `LIMIT 5 OFFSET 5` |
| `WHERE 2000 < price < 5000` | Конструкции двойного сравнения из Python в SQL не работают | `WHERE price BETWEEN 2000 AND 5000` |
| Даты вида `15.03.2026` | Сортируются строково неправильно: `01.10.2026 < 02.09.2026` | Только ISO: `YYYY-MM-DD` |
| Не закрыли `conn.close()` | Файл базы данных может остаться заблокированным процессом | Всегда закрывать в конце работы |


In [ ]:
# Корректное закрытие соединения
conn.close()
print("Соединение с базой данных успешно закрыто.")


Соединение с базой данных успешно закрыто.
